# Read jobs from DynamoDB

Run the cells in order using this project's `.venv` Python kernel. These cells only read AWS resources; they do not change tables or queues.

If a notebook kernel is unavailable, install it with `uv add --dev ipykernel`, then select `.venv/bin/python` in your notebook editor.

The default region is **us-east-1**. Credentials use your existing AWS profile/session; no access keys belong in this notebook. `.env` files are not loaded automatically: inherit the environment from your terminal or set `AWS_PROFILE` below. Do not commit notebook outputs containing personal data.

In [ ]:
import os
from pprint import pprint

import boto3
from boto3.dynamodb.conditions import Key

AWS_REGION = "us-east-1"
AWS_PROFILE = os.environ.get("AWS_PROFILE") or None  # Or your configured profile name.
STACK_NAME = "JobPipeline"  # Change if you deployed JobPipeline-dev, for example.
MAX_ITEMS = 50  # Bound reads while exploring. Use None only to fetch all matches.

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
dynamodb = session.resource("dynamodb")

## Find the deployed tables

Read table names from CloudFormation outputs instead of hardcoding generated names. This requires `cloudformation:DescribeStacks`; reading records also requires DynamoDB read permissions.

In [ ]:
stack = session.client("cloudformation").describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outputs = {entry["OutputKey"]: entry["OutputValue"] for entry in stack["Outputs"]}
jobs_table = dynamodb.Table(outputs["JobsTable"])
classifications_table = dynamodb.Table(outputs["ClassificationsTable"])

print("Region:", AWS_REGION)
print("Jobs:", jobs_table.name)
print("Classifications:", classifications_table.name)

## Read a sample of saved jobs

DynamoDB paginates responses. This helper follows pages while respecting `max_items`. The resource interface returns normal Python values, without `S` / `M` wrappers (numbers may be `Decimal`). A scan has no guaranteed ordering; this sample is not necessarily the newest jobs.

In [ ]:
def read_items(table, operation="scan", *, max_items=50, **kwargs):
    if operation not in {"scan", "query"}:
        raise ValueError("operation must be scan or query")
    if max_items is not None and max_items < 1:
        raise ValueError("max_items must be positive or None")
    pages = table.meta.client.get_paginator(operation).paginate(
        TableName=table.name,
        PaginationConfig={"PageSize": min(max_items, 100) if max_items else 100},
        **kwargs,
    )
    items = []
    for page in pages:
        for item in page.get("Items", []):
            items.append(item)
            if max_items is not None and len(items) >= max_items:
                return items
    return items


jobs = read_items(jobs_table, max_items=MAX_ITEMS)
print(f"Loaded {len(jobs)} jobs (sample, not the table's total count).")
[
    {
        "job_id": item["job_id"],
        "title": item.get("job", {}).get("title"),
        "company": item.get("job", {}).get("company"),
        "location": item.get("job", {}).get("location"),
    }
    for item in jobs
]

In [ ]:
# Inspect a full posting, including its description and application URL.
if jobs:
    pprint(jobs[0], sort_dicts=False)
else:
    print("No jobs found. Check the selected profile, stack, region, and collection logs.")

## Read technical classifications

Query the existing `ByTechSeniority` index for `tech_status = tech` (the stored label for `is_tech = True`). This avoids scanning every classification. Index updates are eventually consistent.

Set `MAX_ITEMS = None` above to retrieve all matching records; this increases read usage and memory consumption. Missing or unfinished classifications are absent from this index.

In [ ]:
tech_classifications = read_items(
    classifications_table,
    "query",
    max_items=MAX_ITEMS,
    IndexName="ByTechSeniority",
    KeyConditionExpression=Key("tech_status").eq("tech"),
)
print(f"Loaded {len(tech_classifications)} technical classifications.")
tech_classifications[:5]

## Fetch postings for those classifications

Use the shared `job_id` to read the corresponding Jobs records. This example reads at most ten postings; results remain in `tech_jobs` for further exploration.

In [ ]:
tech_jobs = []
for classification in tech_classifications[:10]:
    item = jobs_table.get_item(Key={"job_id": classification["job_id"]}).get("Item")
    if item is not None:
        tech_jobs.append({**item, "classification": classification})

[
    {
        "job_id": item["job_id"],
        "title": item["job"].get("title"),
        "company": item["job"].get("company"),
        "location": item["job"].get("location"),
        "category": item["classification"].get("category"),
        "seniority": item["classification"].get("seniority"),
        "application_url": item["job"].get("application_url"),
    }
    for item in tech_jobs
]

## Look up one job by ID

The same ID retrieves its posting and classification. `None` means the corresponding record was not found. You can replace `JOB_ID` with an ID copied from the AWS console.

In [ ]:
JOB_ID = jobs[0]["job_id"] if jobs else ""

if JOB_ID:
    posting = jobs_table.get_item(Key={"job_id": JOB_ID}, ConsistentRead=True).get("Item")
    classification = classifications_table.get_item(
        Key={"job_id": JOB_ID}, ConsistentRead=True
    ).get("Item")
    pprint({"posting": posting, "classification": classification}, sort_dicts=False)
else:
    print("Set JOB_ID to a saved job ID first.")